# Phase 2 Evaluation & Comparison

This notebook visualizes and compares results from Phase 2 (Joint Training) experiments.

**Phase 2 Question:** What frequency preferences are architectural (inductive bias)?

In Phase 2, both the frequency mask AND classifier are trained from scratch.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from frequency.mask import Learnable2DFrequencyMask

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 1. Helper Functions

In [ ]:
def load_mask(results_dir):
    """Load a trained frequency mask."""
    mask_path = results_dir / "learned_mask.pt"
    if not mask_path.exists():
        print(f"Mask not found: {mask_path}")
        return None
    
    mask = Learnable2DFrequencyMask(image_size=224)
    mask.load_state_dict(torch.load(mask_path, map_location='cpu'))
    return mask


def load_history(results_dir):
    """Load training history."""
    history_path = results_dir / "training_history.pt"
    if not history_path.exists():
        print(f"History not found: {history_path}")
        return None
    return torch.load(history_path, map_location='cpu')


def get_mask_weights(mask):
    """Extract mask weights as numpy array."""
    return mask.mask_weights[0, 0].detach().cpu().numpy()


def compute_radial_profile(weights):
    """Compute radial profile (average at each distance from center)."""
    h, w = weights.shape
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    
    r_max = int(np.sqrt(cx ** 2 + cy ** 2))
    radial_profile = []
    for radius in range(r_max):
        ring_mask = (r >= radius) & (r < radius + 1)
        if ring_mask.sum() > 0:
            radial_profile.append(weights[ring_mask].mean())
        else:
            radial_profile.append(0)
    return np.array(radial_profile)


def compute_mask_stats(weights):
    """Compute mask statistics."""
    h, w = weights.shape
    cy, cx = h // 2, w // 2
    
    low_freq = weights[cy-20:cy+20, cx-20:cx+20].mean()
    high_freq = np.mean([weights[:20, :].mean(), weights[-20:, :].mean(),
                         weights[:, :20].mean(), weights[:, -20:].mean()])
    
    return {
        'mean': weights.mean(),
        'std': weights.std(),
        'min': weights.min(),
        'max': weights.max(),
        'low_freq': low_freq,
        'high_freq': high_freq,
        'low_high_ratio': low_freq / high_freq if high_freq != 0 else float('inf')
    }

## 2. Load Phase 2 Results

Specify which architectures have completed Phase 2 training.

In [ ]:
# Base results directory
results_base = Path("results")

# Architectures to analyze (add more as they complete)
phase2_archs = ['resnet18']  # Add: 'alexnet', 'vgg16', 'resnet50' as they complete

# Load all available Phase 2 results
phase2_results = {}

for arch in phase2_archs:
    results_dir = results_base / f"{arch}_phase2"
    if results_dir.exists():
        mask = load_mask(results_dir)
        history = load_history(results_dir)
        if mask is not None:
            phase2_results[arch] = {
                'mask': mask,
                'history': history,
                'weights': get_mask_weights(mask),
                'dir': results_dir
            }
            print(f"Loaded {arch} Phase 2 results")
    else:
        print(f"{arch} Phase 2 not found (not yet trained)")

print(f"\nLoaded {len(phase2_results)} Phase 2 results")

## 3. Training History Visualization

In [ ]:
# Plot training history for each architecture
for arch, data in phase2_results.items():
    history = data['history']
    if history is None:
        continue
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    axes[0].plot(history['epoch'], history['train_loss'], label='Train', linewidth=2)
    axes[0].plot(history['epoch'], history['val_loss'], label='Val', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{arch} - Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history['epoch'], history['train_acc'], label='Train', linewidth=2)
    axes[1].plot(history['epoch'], history['val_acc'], label='Val', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title(f'{arch} - Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle(f'{arch.upper()} Phase 2 Training History', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print final stats
    print(f"{arch}: Final Train Acc: {history['train_acc'][-1]:.2f}% | Final Val Acc: {history['val_acc'][-1]:.2f}%")
    print(f"{arch}: Best Val Acc: {max(history['val_acc']):.2f}%")
    print()

## 4. Phase 2 Learned Masks Visualization

In [ ]:
# Visualize learned masks
n_archs = len(phase2_results)
if n_archs > 0:
    fig, axes = plt.subplots(1, n_archs, figsize=(6 * n_archs, 5))
    if n_archs == 1:
        axes = [axes]
    
    for ax, (arch, data) in zip(axes, phase2_results.items()):
        weights = data['weights']
        stats = compute_mask_stats(weights)
        
        im = ax.imshow(weights, cmap='hot')
        ax.set_title(f"{arch}\nmean={stats['mean']:.3f}, L/H={stats['low_high_ratio']:.2f}")
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)
    
    plt.suptitle('Phase 2 Learned Frequency Masks', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No Phase 2 results to visualize yet")

## 5. Phase 2 Radial Profiles

In [ ]:
# Compare radial profiles across Phase 2 architectures
if len(phase2_results) > 0:
    plt.figure(figsize=(10, 6))
    
    for arch, data in phase2_results.items():
        profile = compute_radial_profile(data['weights'])
        plt.plot(profile, label=arch, linewidth=2)
    
    plt.xlabel('Distance from center (frequency)')
    plt.ylabel('Average mask value')
    plt.title('Phase 2 Radial Frequency Profiles')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Init value')
    plt.show()
else:
    print("No Phase 2 results to compare")

## 6. Phase 1 vs Phase 2 Comparison

Compare the learned masks between frozen classifier (Phase 1) and joint training (Phase 2).

In [ ]:
# Load Phase 1 results for comparison
phase1_archs = ['resnet18', 'alexnet', 'vgg16', 'resnet50']
phase1_results = {}

for arch in phase1_archs:
    results_dir = results_base / arch
    if results_dir.exists():
        mask = load_mask(results_dir)
        if mask is not None:
            phase1_results[arch] = {
                'mask': mask,
                'weights': get_mask_weights(mask),
                'dir': results_dir
            }
            print(f"Loaded {arch} Phase 1 results")

print(f"\nLoaded {len(phase1_results)} Phase 1 results")

In [ ]:
# Compare Phase 1 vs Phase 2 for architectures that have both
common_archs = set(phase1_results.keys()) & set(phase2_results.keys())

for arch in common_archs:
    w1 = phase1_results[arch]['weights']
    w2 = phase2_results[arch]['weights']
    diff = w2 - w1
    
    stats1 = compute_mask_stats(w1)
    stats2 = compute_mask_stats(w2)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    # Phase 1 mask
    im0 = axes[0].imshow(w1, cmap='hot')
    axes[0].set_title(f'Phase 1 (Frozen)\nL/H={stats1["low_high_ratio"]:.2f}')
    axes[0].axis('off')
    plt.colorbar(im0, ax=axes[0])
    
    # Phase 2 mask
    im1 = axes[1].imshow(w2, cmap='hot')
    axes[1].set_title(f'Phase 2 (Joint)\nL/H={stats2["low_high_ratio"]:.2f}')
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1])
    
    # Difference
    vmax = max(abs(diff.min()), abs(diff.max()))
    im2 = axes[2].imshow(diff, cmap='coolwarm', vmin=-vmax, vmax=vmax)
    axes[2].set_title('Difference (P2 - P1)')
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2])
    
    # Radial profiles
    profile1 = compute_radial_profile(w1)
    profile2 = compute_radial_profile(w2)
    axes[3].plot(profile1, label='Phase 1', linewidth=2)
    axes[3].plot(profile2, label='Phase 2', linewidth=2)
    axes[3].set_xlabel('Distance from center')
    axes[3].set_ylabel('Mask value')
    axes[3].set_title('Radial Profiles')
    axes[3].legend()
    axes[3].grid(True, alpha=0.3)
    
    plt.suptitle(f'{arch.upper()}: Phase 1 vs Phase 2 Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"{arch} Phase 1: mean={stats1['mean']:.3f}, L/H ratio={stats1['low_high_ratio']:.2f}")
    print(f"{arch} Phase 2: mean={stats2['mean']:.3f}, L/H ratio={stats2['low_high_ratio']:.2f}")
    print()

if len(common_archs) == 0:
    print("No architectures have both Phase 1 and Phase 2 results yet")

## 7. Cross-Architecture Comparison (Phase 1 Baseline)

In [ ]:
# Compare all Phase 1 masks
if len(phase1_results) > 1:
    n = len(phase1_results)
    fig, axes = plt.subplots(2, n, figsize=(5 * n, 10))
    
    profiles = {}
    
    for i, (arch, data) in enumerate(phase1_results.items()):
        weights = data['weights']
        stats = compute_mask_stats(weights)
        profiles[arch] = compute_radial_profile(weights)
        
        # Mask
        im = axes[0, i].imshow(weights, cmap='hot')
        axes[0, i].set_title(f"{arch}\nL/H={stats['low_high_ratio']:.2f}")
        axes[0, i].axis('off')
        plt.colorbar(im, ax=axes[0, i], fraction=0.046)
        
        # Radial profile
        axes[1, i].plot(profiles[arch], linewidth=2, color='tab:blue')
        axes[1, i].set_xlabel('Frequency')
        axes[1, i].set_ylabel('Mask value')
        axes[1, i].set_title(f'{arch} Radial')
        axes[1, i].grid(True, alpha=0.3)
        axes[1, i].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    
    plt.suptitle('Phase 1 Learned Masks (Frozen Classifier)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Overlay comparison
    plt.figure(figsize=(10, 6))
    for arch, profile in profiles.items():
        plt.plot(profile, label=arch, linewidth=2)
    plt.xlabel('Distance from center (frequency)')
    plt.ylabel('Average mask value')
    plt.title('Phase 1: Radial Profiles Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    plt.show()

## 8. Summary Statistics

In [ ]:
# Summary table
print("=" * 80)
print("SUMMARY: Phase 1 vs Phase 2 Results")
print("=" * 80)
print(f"{'Architecture':<12} {'Phase':<8} {'Mean':<8} {'Std':<8} {'Low Freq':<10} {'High Freq':<10} {'L/H Ratio':<10}")
print("-" * 80)

for arch in sorted(set(list(phase1_results.keys()) + list(phase2_results.keys()))):
    if arch in phase1_results:
        stats = compute_mask_stats(phase1_results[arch]['weights'])
        print(f"{arch:<12} {'P1':<8} {stats['mean']:<8.3f} {stats['std']:<8.3f} {stats['low_freq']:<10.3f} {stats['high_freq']:<10.3f} {stats['low_high_ratio']:<10.2f}")
    
    if arch in phase2_results:
        stats = compute_mask_stats(phase2_results[arch]['weights'])
        print(f"{arch:<12} {'P2':<8} {stats['mean']:<8.3f} {stats['std']:<8.3f} {stats['low_freq']:<10.3f} {stats['high_freq']:<10.3f} {stats['low_high_ratio']:<10.2f}")
    
    print()

## 9. Research Insights

Add your observations here as Phase 2 experiments complete.

In [ ]:
# Phase 1 Results Recap
phase1_summary = {
    'resnet18': {'baseline': 65.70, 'final': 67.24, 'change': '+1.54%'},
    'alexnet': {'baseline': 52.48, 'final': 47.48, 'change': '-5.00%'},
    'vgg16': {'baseline': 69.48, 'final': 65.77, 'change': '-3.70%'},
    'resnet50': {'baseline': 74.18, 'final': 74.56, 'change': '+0.38%'}
}

print("Phase 1 Accuracy Summary (Frozen Classifier):")
print("=" * 50)
for arch, data in phase1_summary.items():
    print(f"{arch:<12}: {data['baseline']:.2f}% -> {data['final']:.2f}% ({data['change']})")

print("\nKey Insight: ResNet architectures IMPROVE with frequency filtering,")
print("while AlexNet and VGG-16 DEGRADE. This suggests skip connections")
print("enable more robust frequency-selective processing.")